# 🚀 DataMate (Zalo AI Challenge 2026) — Kaggle Notebook Runner

Notebook này được tối ưu hóa đặc biệt cho môi trường **Kaggle (GPU T4 x 2)** để chạy mô hình mạnh nhất **`Qwen/Qwen2.5-Coder-14B-Instruct` ở chế độ 8-bit (INT8)**.

### ⚙️ Cài đặt bắt buộc trên giao diện Kaggle (Notebook Settings bên phải):
1. **Accelerator**: Chọn **GPU T4 x 2** (2 x 15GB VRAM = 30GB VRAM, hệ thống tự động phân tải đều sang cả 2 card).
2. **Internet**: Chọn **Internet on** (Bắt buộc để nạp thư viện và tải trọng số mô hình từ Hugging Face).
3. **Persistence**: Chọn **Variables and Files**.
4. **Dữ liệu (+ Add Input)**: Vì `competition_data` nằm trong `.gitignore` (không đưa lên GitHub), bạn bấm **+ Add Input** ở góc trên bên phải để thêm Dataset chứa `competition_data` (hoặc `train` và `public_test`). Notebook sẽ **tự động quét và liên kết dữ liệu**.

---
### 💡 Bạn chỉ cần tùy chỉnh tại **Ô Cấu Hình (Cell 1)**:
- **Chọn khoảng câu trên tập Train**: Đặt `TRAIN_RANGE = '1-50'` hoặc `'51-100'` (chạy chính xác từ câu X đến câu Y không random).
- **Chọn số câu trên Public Test**: `PUBLIC_RANGE = '1-100'` hoặc `PUBLIC_BATCH_SIZE = -1` (đủ 1.000 câu).
- **Thứ tự chạy**: Đặt `PRESERVE_ORDER = True` để chạy tuần tự theo đúng thứ tự file `questions.jsonl`.

In [ ]:
# ==============================================================================
# 🎯 CẤU HÌNH TẬP TRUNG (CHỌN KHOẢNG CÂU & CHẾ ĐỘ CHẠY TẠI ĐÂY)
# ==============================================================================

# 1. ĐƯỜNG DẪN DỮ LIỆU TRÊN KAGGLE (/kaggle/input/...):
# (Nếu bạn Add Dataset riêng vào Kaggle, hãy điền đường dẫn thư mục chứa train và public_test)
# Mặc định hệ thống cũng sẽ tự động quét phát hiện dữ liệu trong mọi thư mục /kaggle/input/.
KAGGLE_DATA_DIR = "/kaggle/input/datamate-data/competition_data"

# 2. CẤU HÌNH CHỌN CÂU TRÊN TẬP TRAIN (BƯỚC 5):
# 👉 CÁCH 1 (CHỌN CHÍNH XÁC TỪ CÂU NÀO ĐẾN CÂU NÀO):
#    Điền khoảng "START-END" (ví dụ: "1-50", "51-100", "101-200") -> Chạy đúng khoảng này, KHÔNG random!
# 👉 CÁCH 2 (LẤY MẪU NGẪU NHIÊN): Đặt TRAIN_RANGE = None và điền TRAIN_SAMPLE_SIZE = 30
# 👉 CÁCH 3 (BỎ QUA TẬP TRAIN): Đặt TRAIN_RANGE = None và TRAIN_SAMPLE_SIZE = 0
TRAIN_RANGE = "1-30"              # Ví dụ: "1-50", "51-100", "101-200", hoặc None
TRAIN_SAMPLE_SIZE = 30            # Dùng khi TRAIN_RANGE = None (đặt 0 nếu bỏ qua tập train)
TRAIN_MODE = "random"             # "random" (ngẫu nhiên phân tầng) hoặc "sequential" (từ câu 1 đến N)
TRAIN_SEED = 42

# 3. CẤU HÌNH SỐ LƯỢNG CÂU & THỨ TỰ CHẠY TRÊN PUBLIC TEST (BƯỚC 6):
# - PUBLIC_RANGE: Khoảng câu hỏi muốn chạy trên public_test (ví dụ: '1-100', '101-200', hoặc None)
#     • Nếu điền '1-100' -> Chạy chính xác từ câu 1 đến câu 100
#     • Nếu đặt None     -> Chạy theo PUBLIC_BATCH_SIZE bên dưới
PUBLIC_RANGE = None

# - PUBLIC_BATCH_SIZE: Số câu bạn muốn chạy (-1 hoặc 1000 = toàn bộ 1000 câu; hoặc điền 100, 500...)
PUBLIC_BATCH_SIZE = -1

# - PRESERVE_ORDER: Thứ tự chạy các câu hỏi:
#     • True : Chạy TUẦN TỰ từ câu 1 đến câu 1000 đúng theo thứ tự trong questions.jsonl (KHÔNG random, KHÔNG đảo shop)
#     • False: Gom nhóm các câu hỏi cùng shop để chạy liên tiếp (tối ưu tốc độ RAM Cache cao nhất)
PRESERVE_ORDER = True

# 4. CẤU HÌNH MÔ HÌNH LLM:
MODEL_ID = "Qwen/Qwen2.5-Coder-14B-Instruct"   # Hoặc 'Qwen/Qwen2.5-Coder-7B-Instruct'
QUANTIZATION = "8bit"                          # "8bit" (mạnh nhất trên GPU T4x2) hoặc "4bit"

# 5. SIÊU THAM SỐ PIPELINE:
TAU_STAR = 0.50                   # Ngưỡng tin cậy hiệu chuẩn tối ưu cho Penalized Accuracy (PA)
TIMEOUT_S = 30                    # Thời gian chờ tối đa khi chạy code trong sandbox (giây)
MAX_NEW_TOKENS = 1536             # Token tối đa cho sinh mã Python

# 6. ĐƯỜNG DẪN MÃ NGUỒN VÀ KẾT QUẢ:
REPO_URL = "https://github.com/Long-uit/datamate.git"
REPO_BRANCH = "long-commit"       # Nhánh làm việc chính thức ('long-commit' hoặc 'phat_commit')
PROJECT_ROOT = "/kaggle/working/datamate"
RUN_DIR = "/kaggle/working/results"
SUBMISSION_PATH = "/kaggle/working/submission.jsonl"

if PUBLIC_RANGE:
    pub_desc = f"từ câu {PUBLIC_RANGE}"
else:
    target_pub = 1000 if (PUBLIC_BATCH_SIZE is None or int(PUBLIC_BATCH_SIZE) <= 0 or int(PUBLIC_BATCH_SIZE) >= 1000) else int(PUBLIC_BATCH_SIZE)
    pub_desc = f"{target_pub} câu"
train_desc = f"từ câu {TRAIN_RANGE}" if TRAIN_RANGE else f"{TRAIN_SAMPLE_SIZE} câu ({TRAIN_MODE})"
print(f"✅ Đã thiết lập: MODEL={MODEL_ID} ({QUANTIZATION}) | PRESERVE_ORDER={PRESERVE_ORDER}")
print(f"🎯 Tập Train: {train_desc} | Public Test: {pub_desc}")

In [ ]:
# ==============================================================================
# BƯỚC 1: XÁC NHẬN CẤU HÌNH PHẦN CỨNG GPU TRÊN KAGGLE
# ==============================================================================
import torch

!nvidia-smi

if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"\n✅ Đã phát hiện {gpu_count} GPU CUDA khả dụng:")
    for i in range(gpu_count):
        name = torch.cuda.get_device_name(i)
        mem = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
        print(f"   • GPU {i}: {name} ({mem:.1f} GB VRAM)")
    
    if gpu_count >= 2:
        print("🎉 CẤU HÌNH LÝ TƯỞNG: Đang chạy trên GPU T4 x 2! Mô hình 14B 8-bit sẽ tự động chia tải qua cả 2 card.")
    else:
        print("ℹ️ Đang chạy trên 1 GPU. Nếu là P100 (16GB) hoặc T4 đơn, bạn có thể cân nhắc QUANTIZATION='4bit' nếu VRAM chật.")
else:
    print("❌ CẢNH BÁO: Chưa bật GPU! Hãy vào Notebook Settings (bên phải) -> Accelerator -> Chọn 'GPU T4 x 2'.")

In [ ]:
# ==============================================================================
# BƯỚC 2: CLONE MÃ NGUỒN DATAMATE VÀ CÀI ĐẶT THƯ VIỆN BỔ SUNG
# ==============================================================================
import os
import subprocess

if not os.path.exists(PROJECT_ROOT):
    print(f"🔄 Đang clone repository từ {REPO_URL} (ưu tiên nhánh: {REPO_BRANCH})...")
    ret = subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, PROJECT_ROOT])
    if ret.returncode != 0:
        print(f"⚠️ Không tìm thấy nhánh {REPO_BRANCH}, đang thử clone nhánh phat_commit...")
        ret2 = subprocess.run(["git", "clone", "-b", "phat_commit", REPO_URL, PROJECT_ROOT])
        if ret2.returncode != 0:
            print("⚠️ Đang clone nhánh mặc định...")
            !git clone {REPO_URL} {PROJECT_ROOT}
else:
    print(f"🔄 Thư mục {PROJECT_ROOT} đã tồn tại, đang cập nhật mã nguồn mới nhất...")
    %cd {PROJECT_ROOT}
    !git fetch --all
    !git checkout {REPO_BRANCH} || git checkout phat_commit || true
    !git pull origin {REPO_BRANCH} || git pull origin phat_commit || true

%cd {PROJECT_ROOT}
!git status -sb

print("\n📦 Đang cài đặt các thư viện cần thiết...")
!pip install -q -r colab/requirements-colab.txt
!pip install -q -e . --no-deps
print("✅ Hoàn tất thiết lập môi trường DataMate trên Kaggle!")

In [ ]:
# ==============================================================================
# BƯỚC 3: TỰ ĐỘNG PHÁT HIỆN VÀ ĐỒNG BỘ BỘ DỮ LIỆU COMPETITION_DATA
# ==============================================================================
import os
import shutil
from pathlib import Path

%cd {PROJECT_ROOT}
target_data_dir = Path(PROJECT_ROOT) / "competition_data"
target_data_dir.mkdir(parents=True, exist_ok=True)

# Tự động phát hiện và liên kết thư mục train và public_test từ /kaggle/input/
for split in ["train", "public_test"]:
    dst_split = target_data_dir / split
    if dst_split.exists() and (dst_split / "questions.jsonl").exists():
        continue
    found_split_dir = None
    
    # 1. Thử từ KAGGLE_DATA_DIR
    candidate = Path(KAGGLE_DATA_DIR) / split
    if candidate.exists() and (candidate / "questions.jsonl").exists():
        found_split_dir = candidate
    
    # 2. Quét đệ quy toàn bộ thư mục trong /kaggle/input/
    if not found_split_dir and Path("/kaggle/input").exists():
        for q_file in Path("/kaggle/input").rglob("questions.jsonl"):
            if q_file.parent.name == split:
                found_split_dir = q_file.parent
                break
            elif split == "public_test" and (q_file.parent / "rulebook.md").exists() and not (q_file.parent / "answers.jsonl").exists():
                found_split_dir = q_file.parent
                break
    
    if found_split_dir and not dst_split.exists():
        print(f"🔗 Đã phát hiện {split} tại: {found_split_dir} -> Đang liên kết...")
        try:
            os.symlink(str(found_split_dir), str(dst_split))
        except Exception:
            shutil.copytree(str(found_split_dir), str(dst_split))

# Thống kê số lượng câu hỏi và cửa hàng sẵn sàng
print("\n📊 TỔNG HỢP TRẠNG THÁI DỮ LIỆU SẴN SÀNG:")
for split in ["train", "public_test"]:
    s_dir = target_data_dir / split
    q_f = s_dir / "questions.jsonl"
    a_f = s_dir / "answers.jsonl"
    d_d = s_dir / "data"
    if q_f.exists():
        n_q = sum(1 for line in open(q_f, encoding="utf-8") if line.strip())
        n_biz = len([p for p in d_d.iterdir() if p.is_dir()]) if d_d.exists() else 0
        ans_status = "Có đáp án chuẩn answers.jsonl" if a_f.exists() else "Tập test ẩn đáp án"
        print(f"  • [{split:<11}] {n_q:>4} câu hỏi | {n_biz:>3} doanh nghiệp | {ans_status}")
    else:
        print(f"  ⚠️ [{split:<11}] Chưa tìm thấy {q_f}.")
        print(f"      👉 Hãy bấm '+ Add Input' bên phải để thêm Dataset competition_data vào Kaggle!")

In [ ]:
# ==============================================================================
# BƯỚC 4: KIỂM THỬ NHANH HỆ THỐNG (UNIT TESTS & DRY-RUN KHÔNG DÙNG GPU)
# ==============================================================================
%cd {PROJECT_ROOT}

print("1️⃣ Chạy kiểm thử cấu hình & tính tương thích...")
!python -m pytest tests/test_config.py -q || true

print("\n2️⃣ Chạy thử nghiệm phân tích White-Box 1 câu hỏi (Dry-Run)...)")
if (target_data_dir / "train" / "questions.jsonl").exists():
    dry_cmd = "python tests/inspect_pipeline.py --data_dir competition_data/train --random --dry_run"
    !{dry_cmd}
elif (target_data_dir / "public_test" / "questions.jsonl").exists():
    dry_cmd = "python tests/inspect_pipeline.py --data_dir competition_data/public_test --limit 1 --dry_run"
    !{dry_cmd}
else:
    print("⚠️ Chưa có dữ liệu competition_data. Hãy kiểm tra lại Bước 3!")

print("✅ Khung pipeline chạy thông suốt (Dry-Run hoàn tất)!")

In [ ]:
# ==============================================================================
# BƯỚC 5: ĐÁNH GIÁ CHẤT LƯỢNG MÔ HÌNH 14B TRÊN TẬP TRAIN (THEO KHOẢNG CÂU HOẶC MẪU)
# ==============================================================================
import os
from pathlib import Path
from IPython.display import Image, display

%cd {PROJECT_ROOT}
train_eval_dir = Path(RUN_DIR) / "train_eval"
train_eval_dir.mkdir(parents=True, exist_ok=True)

train_pred_file = train_eval_dir / "train_predictions.jsonl"
train_log_dir = train_eval_dir / "logs"
train_rep_file = train_eval_dir / "evaluation_report.json"
train_md_file = train_eval_dir / "evaluation_report.md"
train_chart_file = train_eval_dir / "evaluation_dashboard.png"

if not (Path("competition_data/train/questions.jsonl").exists()):
    print("⏭️ Không tìm thấy competition_data/train/questions.jsonl. Tự động bỏ qua Bước 5.")
elif not TRAIN_RANGE and (TRAIN_SAMPLE_SIZE is not None and int(TRAIN_SAMPLE_SIZE) == 0):
    print("⏭️ Đã cấu hình bỏ qua kiểm định trên tập TRAIN (TRAIN_SAMPLE_SIZE = 0).")
else:
    flags = []
    if TRAIN_RANGE:
        flags.append(f"--range {TRAIN_RANGE}")
        print(f"🎯 Đang đánh giá CHÍNH XÁC từ câu {TRAIN_RANGE} của tập TRAIN...")
    elif TRAIN_MODE == "random" and int(TRAIN_SAMPLE_SIZE) > 0:
        flags.append(f"--sample {TRAIN_SAMPLE_SIZE} --stratified")
        print(f"🎯 Đang lấy mẫu ngẫu nhiên phân tầng {TRAIN_SAMPLE_SIZE} câu...")
    elif int(TRAIN_SAMPLE_SIZE) > 0:
        flags.append(f"--limit {TRAIN_SAMPLE_SIZE}")
        print(f"🎯 Đang chạy tuần tự {TRAIN_SAMPLE_SIZE} câu đầu tiên...")
    
    if PRESERVE_ORDER:
        flags.append("--preserve_order")
    extra_flags = " ".join(flags)

    print(f"🚀 Bắt đầu đánh giá mô hình {MODEL_ID} ({QUANTIZATION}) trên tập TRAIN...")
    train_cmd = (
        f"python tests/inspect_pipeline.py "
        f"--batch "
        f"--with_llm "
        f"--data_dir competition_data/train "
        f"--model_id '{MODEL_ID}' "
        f"--quantization '{QUANTIZATION}' "
        f"--seed {TRAIN_SEED} "
        f"--tau_star {TAU_STAR} "
        f"--timeout {TIMEOUT_S} "
        f"--max_new_tokens {MAX_NEW_TOKENS} "
        f"--no_fill_abstain "
        f"--output_file '{train_pred_file}' "
        f"--log_dir '{train_log_dir}' "
        f"--report_file '{train_rep_file}' "
        f"--md_file '{train_md_file}' "
        f"--chart_file '{train_chart_file}' "
        f"{extra_flags}"
    )
    !{train_cmd}

if train_chart_file.exists():
    print("\n📊 Biểu đồ trực quan kết quả đánh giá (Evaluation Dashboard):")
    display(Image(filename=str(train_chart_file)))

In [ ]:
# ==============================================================================
# BƯỚC 6: SUY LUẬN CHÍNH THỨC TRÊN PUBLIC TEST (XUẤT SUBMISSION.JSONL)
# ==============================================================================
# Chạy theo thứ tự tuần tự từ câu 1 đến câu 1000 (nếu PRESERVE_ORDER = True).
# Hỗ trợ cờ --resume: nếu dừng hoặc bị ngắt, chạy lại sẽ tự động làm tiếp từ câu chưa xong.
import os
from pathlib import Path

%cd {PROJECT_ROOT}
Path(SUBMISSION_PATH).parent.mkdir(parents=True, exist_ok=True)
pub_log_dir = Path(RUN_DIR) / "public_test_logs"
pub_log_dir.mkdir(parents=True, exist_ok=True)

if not (Path("competition_data/public_test/questions.jsonl").exists()):
    print("❌ LỖI: Chưa có competition_data/public_test/questions.jsonl! Hãy kiểm tra lại Bước 3.")
else:
    flags = ["--resume"]
    if PUBLIC_RANGE:
        flags.append(f"--range {PUBLIC_RANGE}")
    elif PUBLIC_BATCH_SIZE is not None and 0 < int(PUBLIC_BATCH_SIZE) < 1000:
        flags.append(f"--limit {PUBLIC_BATCH_SIZE}")
    if PRESERVE_ORDER:
        flags.append("--preserve_order")
    extra_flags = " ".join(flags)

    if PUBLIC_RANGE:
        target_count = f"Khoảng câu hỏi {PUBLIC_RANGE}"
    else:
        target_count = "Toàn bộ 1.000 câu" if (PUBLIC_BATCH_SIZE is None or int(PUBLIC_BATCH_SIZE) <= 0 or int(PUBLIC_BATCH_SIZE) >= 1000) else f"{PUBLIC_BATCH_SIZE} câu đầu tiên"
    print(f"🚀 Bắt đầu suy luận nộp bài trên public_test: {target_count}...")
    print(f"📌 Mô hình: {MODEL_ID} ({QUANTIZATION}) | Thứ tự: {'TUẦN TỰ từ câu 1 -> 1000' if PRESERVE_ORDER else 'GOM NHÓM theo shop'}")
    
    predict_cmd = (
        f"python predict.py "
        f"--data_dir competition_data/public_test "
        f"--model_id '{MODEL_ID}' "
        f"--quantization '{QUANTIZATION}' "
        f"--output_file '{SUBMISSION_PATH}' "
        f"--log_dir '{pub_log_dir}' "
        f"--tau_star {TAU_STAR} "
        f"--timeout {TIMEOUT_S} "
        f"--seed 42 "
        f"{extra_flags}"
    )
    !{predict_cmd}

    print(f"✅ Quá trình suy luận hoàn tất! Kết quả đã ghi tại: {SUBMISSION_PATH}")

In [ ]:
# ==============================================================================
# BƯỚC 7: THẨM ĐỊNH TỰ ĐỘNG & HƯỚNG DẪN TẢI FILE SUBMISSION.JSONL NỘP BÀI
# ==============================================================================
import json
from collections import Counter
from pathlib import Path
from datamate.output_abstention.submission import validate_submission

%cd {PROJECT_ROOT}
submission_file = Path(SUBMISSION_PATH)
questions_file = Path(PROJECT_ROOT) / "competition_data/public_test/questions.jsonl"

if not submission_file.exists():
    print(f"❌ Chưa tìm thấy tệp {submission_file}. Hãy kiểm tra lại Bước 6!")
elif not questions_file.exists():
    print(f"❌ Chưa tìm thấy tệp {questions_file} để đối chiếu.")
else:
    all_qids = [json.loads(line)["id"] for line in open(questions_file, encoding="utf-8") if line.strip()]
    done_records = {}
    for line in open(submission_file, encoding="utf-8"):
        if line.strip():
            rec = json.loads(line)
            done_records[rec["id"]] = rec

    print(f"📌 Tiến độ hồ sơ nộp bài: {len(done_records)} / {len(all_qids)} câu hỏi ({100*len(done_records)/len(all_qids):.1f}%)")

    errors = validate_submission(submission_file, questions_file)
    format_errors = [e for e in errors if not e.startswith("Thiếu ")]
    missing_errors = [e for e in errors if e.startswith("Thiếu ")]

    if not format_errors:
        print("🎉 CẤU TRÚC ĐÁP ÁN: 100% câu hỏi đã hoàn thành đều đạt chuẩn cấu trúc của Ban Tổ Chức!")
    else:
        print(f"⚠️ Phát hiện {len(format_errors)} vấn đề định dạng:")
        for err in format_errors[:10]:
            print(f"  • {err}")

    if not missing_errors:
        print("🎉 TOÀN VẸN: Đã hoàn thành đủ 1.000 / 1.000 câu hỏi, sẵn sàng nộp bài chấm điểm chính thức!")
    else:
        print(f"ℹ️ Trạng thái số lượng câu: {missing_errors[0]}")

    type_counts = Counter(r.get("type", "unknown") for r in done_records.values())
    print("\n📊 Phân bố các dạng đáp án trong submission.jsonl:")
    for t_name, count in type_counts.most_common():
        print(f"  • {t_name:<16}: {count:>5} câu ({100*count/max(1, len(done_records)):5.1f}%)")

    print("\n📥 HƯỚNG DẪN TẢI FILE NỘP BÀI TỪ KAGGLE:")
    print(f"1. Nhìn sang cột bên phải (Notebook Sidebar) -> Mục 'Output'.")
    print(f"2. Tìm tệp '{submission_file.name}' trong thư mục /kaggle/working/.")
    print("3. Bấm vào biểu tượng 3 chấm (...) bên cạnh tệp và chọn 'Download'.")